#### Transform Refunds Tables
- Extract specific portion of string from refund_reason using split function
- Extract specific portion of string from refund_reason using regexp_extract function
- Extract the date and time from refund_timestamp
- Write transformed data into the silver layer

#### 0. Create the bronze refunds table from Azure SQL

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gizmobox.bronze;

CREATE TABLE IF NOT EXISTS gizmobox.bronze.refunds AS
SELECT
    refund_id,
    payment_id,
    refund_timestamp,
    refund_amount,
    refund_reason
FROM asql_gizmobox_db_catalog.dbo.refunds;

In [0]:
%sql
SELECT
    *
FROM gizmobox.bronze.refunds
ORDER BY refund_id;

#### 1. Extract specific portion of string from refund_reason using split function

https://docs.databricks.com/aws/en/sql/language-manual/functions/split


In [0]:
%sql
SELECT 
    *,
    split(refund_reason, ':')[0] AS refund_reason,
    split(refund_reason, ':')[1] AS refund_source
FROM gizmobox.bronze.refunds

#### 2. Extract specific portion of string from refund_reason using regexp_extract function
- 
- https://docs.databricks.com/aws/en/sql/language-manual/functions/regexp_extract
- https://regexr.com/


In [0]:
%sql
SELECT 
    *,
    regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason1,
    regexp_extract(refund_reason, '([^:]+)$', 1) AS refund_score
FROM gizmobox.bronze.refunds

#### 3. Extract the date and time from refund_timestamp

In [0]:
%sql
SELECT 
    refund_id,
    payment_id,
    CAST(date_format(refund_timestamp,'yyyy-MM-dd') AS DATE) AS refund_date,
    date_format(refund_timestamp,'HH:mm:ss') AS refund_time,
    refund_amount,
    regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
    regexp_extract(refund_reason, '([^:]+)$', 1) AS refund_source
FROM gizmobox.bronze.refunds

#### 4. Write transformed data into the silver layer

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.refunds AS
SELECT 
    refund_id,
    payment_id,
    CAST(date_format(refund_timestamp,'yyyy-MM-dd') AS DATE) AS refund_date,
    date_format(refund_timestamp,'HH:mm:ss') AS refund_time,
    refund_amount,
    regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
    regexp_extract(refund_reason, '([^:]+)$', 1) AS refund_source
FROM gizmobox.bronze.refunds

In [0]:
%sql
SELECT * FROM gizmobox.silver.refunds